# 06 Final Optimized TabM

Train the optimized TabM and generate the first final submission. Default: longer internal epoch selection, original features and three seeds. Report: Section 3.4 and Appendix A.


## 1. Setup
Set the data path, choose a runtime and install the dependencies used below. The `quick` profile checks the workflow with a small budget; use `report` for the report experiments. Place the four original CSV files in one folder.


In [ ]:
from pathlib import Path
# Use the same data directory and run name in all notebooks.
PROFILE = globals().get('PROFILE', 'report')  # quick: smoke test; report: full training
DATA_SOURCE = globals().get('DATA_SOURCE', 'drive')  # drive, upload or local
DATA_DIR = Path(globals().get('DATA_DIR', '/content/drive/MyDrive/bt-4012-competition-2026'))
OUTPUT_ROOT = Path(globals().get('OUTPUT_ROOT', '/content/drive/MyDrive/BT4012_Final_Report_English'))
RUN_NAME = globals().get('RUN_NAME', 'final_report_english_v1')
INSTALL_DEPENDENCIES = globals().get('INSTALL_DEPENDENCIES', True)
EXPECTED_COMPETITION = globals().get('EXPECTED_COMPETITION', True)
assert PROFILE in ['quick', 'report']
OUT = OUTPUT_ROOT / RUN_NAME / PROFILE


In [ ]:
import sys, subprocess
# Keep the torch/CUDA version supplied by Colab.
PACKAGES = ['numpy>=1.26', 'pandas>=2.0', 'scipy>=1.11', 'scikit-learn>=1.6', 'joblib', 'tabm==0.0.3', 'rtdl-num-embeddings==0.0.12']
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], check=True)


In [ ]:
CSV_FILES = ['train.csv', 'test.csv', 'txs_edgelist.csv', 'sample_submission.csv']

if DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
elif DATA_SOURCE == 'upload':
    from google.colab import files
    DATA_DIR = Path('/content/bt4012_data')
    DATA_DIR.mkdir(exist_ok=True)
    missing = [name for name in CSV_FILES if not (DATA_DIR / name).exists()]
    if missing:
        print('Upload the missing files:', missing)
        for name, content in files.upload().items():
            if name not in missing:
                raise ValueError('Unexpected file: ' + name)
            (DATA_DIR / name).write_bytes(content)
    if str(OUTPUT_ROOT).startswith('/content/drive/'):
        OUTPUT_ROOT = Path('/content/BT4012_Final_Report_English')
        OUT = OUTPUT_ROOT / RUN_NAME / PROFILE
elif DATA_SOURCE != 'local':
    raise ValueError(DATA_SOURCE)
for name in CSV_FILES:
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(DATA_DIR / name)
OUT.mkdir(parents=True, exist_ok=True)
print('Mode:', PROFILE, '| Output:', OUT)


Mounted at /content/drive
Mode: report | Output: /content/drive/MyDrive/BT4012_Final_Report_English/final_report_english_v1/report


## 2. Required functions
Only the data structures, models and helpers used by this notebook are included below.


In [ ]:
from torch.nn import functional as F
from pathlib import Path
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from scipy.sparse.csgraph import connected_components
from scipy.sparse import coo_matrix
from collections import deque
import gc
import hashlib
import joblib
import json
from torch import nn
import numpy as np
import pandas as pd
import random
from sklearn.metrics import roc_auc_score
import torch
import os
torch.set_num_threads(min(4, os.cpu_count() or 1))

ENGINE_HASH = '3661498eb4e7e545253aeae84b480c6d4c349dc7e125920bf97989a5e96c0840'

FEATURES = [f'feat_{i}' for i in range(1, 166)]

TABM = dict(k=32, n_blocks=2, d_block=512, dropout=0.1, n_bins=48, d_embedding=16, learning_rate=0.002, weight_decay=0.0003)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def serial(x):
    if isinstance(x, (np.integer, np.floating)):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, Path):
        return str(x)
    raise TypeError(type(x).__name__)

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=serial, allow_nan=False))
    temp.replace(path)

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=serial).encode()).hexdigest()

def array_hash(x):
    return hashlib.sha256(np.ascontiguousarray(x).tobytes()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 ** 2), b''):
            h.update(block)
    return h.hexdigest()

def seed_all(seed):
    """Set random seeds before constructing and training a model."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def metrics(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    if len(y) != len(p) or not len(y) or (not np.isfinite(p).all()):
        raise ValueError('Empty, nonfinite or unaligned predictions')
    if not np.isin(y, [0, 1]).all():
        raise ValueError('Unknown labels cannot be scored')
    return dict(n=len(y), positives=int(y.sum()), auc=float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else None, ap=float(average_precision_score(y, p)) if len(np.unique(y)) == 2 else None)

def profiles(mode):
    """Use a small budget for a smoke test or the report budget for full experiments."""
    if mode not in ['quick', 'report']:
        raise ValueError(mode)
    return dict(mode=mode, seeds=[42] if mode == 'quick' else [17, 42, 73], cutoffs=[21] if mode == 'quick' else [7, 10, 14, 17, 21], max_epochs=2 if mode == 'quick' else 100, patience=1 if mode == 'quick' else 10, trees=30 if mode == 'quick' else None, batch_size=256)

class Dataset:
    """Read the four CSV files and build the transaction graph."""

    def __init__(self, data_dir, expected_competition=True):
        self.data_dir = Path(data_dir)
        self.train = pd.read_csv(self.data_dir / 'train.csv')
        self.test = pd.read_csv(self.data_dir / 'test.csv')
        self.sample = pd.read_csv(self.data_dir / 'sample_submission.csv')
        raw_edges = pd.read_csv(self.data_dir / 'txs_edgelist.csv')
        self.duplicate_edges = int(raw_edges.duplicated().sum())
        self.edges = raw_edges.drop_duplicates(['txId1', 'txId2']).reset_index(drop=True)
        if list(self.train.columns) != ['txId', 'time_step', *FEATURES, 'label']:
            raise ValueError('Unexpected train columns/order')
        if list(self.test.columns) != ['index', 'txId', 'time_step', *FEATURES]:
            raise ValueError('Unexpected test columns/order')
        if list(self.sample.columns) != ['index', 'target']:
            raise ValueError('Submission schema')
        if self.train.txId.duplicated().any() or self.test.txId.duplicated().any():
            raise ValueError('Duplicate IDs')
        if set(self.train.txId) & set(self.test.txId):
            raise ValueError('Overlapping IDs')
        if not np.array_equal(self.sample['index'], self.test['index']):
            raise ValueError('Submission order')
        if not self.test['index'].is_unique:
            raise ValueError('Duplicate submission IDs')
        if not self.train.label.dropna().isin([0, 1]).all():
            raise ValueError('Labels must be 0/1/blank')
        for df in [self.train, self.test]:
            if not np.isfinite(df[FEATURES].to_numpy()).all():
                raise ValueError('Nonfinite raw features')
        if expected_competition:
            assert (len(self.train), len(self.test), len(self.edges)) == (141772, 15329, 234355)
            assert self.train.time_step.between(1, 35).all() and self.test.time_step.between(36, 49).all()
        self.ids = np.unique(np.r_[self.train.txId, self.test.txId, self.edges.to_numpy().ravel()])
        self.src = np.searchsorted(self.ids, self.edges.txId1)
        self.dst = np.searchsorted(self.ids, self.edges.txId2)
        n = len(self.ids)
        self.outgoing = coo_matrix((np.ones(len(self.src)), (self.src, self.dst)), shape=(n, n)).tocsr()
        self.incoming = self.outgoing.T.tocsr()
        self.indeg = np.diff(self.incoming.indptr)
        self.outdeg = np.diff(self.outgoing.indptr)
        self.nc, self.components = connected_components(self.outgoing, directed=False)
        all_rows = pd.concat([self.train, self.test], ignore_index=True).set_index('txId').reindex(self.ids)
        self.node_raw = all_rows[FEATURES].to_numpy(np.float32)
        self.has_features = np.isfinite(self.node_raw).all(1)
        self.node_y = all_rows.label.fillna(-1).to_numpy(np.int64)
        observed = all_rows.time_step.to_numpy(float)
        self.steps = np.full(n, -1, np.int64)
        for c in range(self.nc):
            member = self.components == c
            times = np.unique(observed[member & np.isfinite(observed)])
            if len(times) != 1:
                raise ValueError('Component does not identify exactly one snapshot')
            self.steps[member] = int(times[0])
        if np.any(self.steps[self.src] != self.steps[self.dst]):
            raise ValueError('Cross-snapshot edge')
        q = deque(np.flatnonzero(self.indeg == 0).tolist())
        deg = self.indeg.copy()
        self.order = []
        while q:
            v = q.popleft()
            self.order.append(v)
            for w in self.outgoing.indices[self.outgoing.indptr[v]:self.outgoing.indptr[v + 1]]:
                deg[w] -= 1
                if deg[w] == 0:
                    q.append(int(w))
        if len(self.order) != n:
            raise ValueError('Graph is not a DAG')
        # Unknown labels are graph context, not negative examples.
        self.frame = self.train.loc[self.train.label.notna()].copy().reset_index(drop=True)
        self.frame.label = self.frame.label.astype(np.int64)
        self.pos = np.searchsorted(self.ids, self.frame.txId)
        self.testpos = np.searchsorted(self.ids, self.test.txId)
        self.node_x = np.column_stack([self.node_raw, self.steps, self.indeg, self.outdeg]).astype(np.float32)
        self.raw = self.node_x[self.pos]
        self.y = self.frame.label.to_numpy(np.int64)
        self._extras = {}
        self.data_hashes = {name: file_hash(self.data_dir / name) for name in ['train.csv', 'test.csv', 'txs_edgelist.csv', 'sample_submission.csv']}
        self.identity = fingerprint(self.data_hashes)

    def split(self, cut, horizon=14):
        t = self.frame.time_step.to_numpy()
        a = np.flatnonzero(t <= cut)
        b = np.flatnonzero((t > cut) & (t <= cut + horizon))
        if not len(a) or not len(b):
            raise ValueError('Empty temporal split')
        return (a, b)


In [ ]:
def prepare_num(A, B, scaler='quantile'):
    """Fit the imputer, constant-column mask and scaler using training rows only."""
    # Fit every transformation using the current training fold.
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    X = imputer.fit_transform(A).astype(np.float32)
    V = imputer.transform(B).astype(np.float32)
    keep = np.ptp(X, axis=0) > 0
    if not keep.any():
        raise ValueError('All numerical inputs constant')
    X = X[:, keep]
    V = V[:, keep]
    if scaler == 'quantile':
        transform = QuantileTransformer(n_quantiles=max(10, min(len(X) // 30, 1000)), output_distribution='normal', subsample=10 ** 9, random_state=0)
        noise = np.random.default_rng(0).normal(0, 1e-05, X.shape).astype(np.float32)
        transform.fit(X + noise)
    elif scaler == 'robust':
        transform = RobustScaler().fit(X)
    elif scaler == 'standard':
        transform = StandardScaler().fit(X)
    elif scaler == 'none':
        transform = None
    else:
        raise ValueError(scaler)
    bundle = (imputer, keep, transform)
    return (apply_num(A, bundle), apply_num(B, bundle), bundle)

def apply_num(X, bundle):
    """Apply a saved numeric transformation without refitting it."""
    imputer, keep, transform = bundle
    X = imputer.transform(X).astype(np.float32)[:, keep]
    return (transform.transform(X) if transform is not None else X).astype(np.float32)

def make_inputs(data, fit, query, variant='raw', scaler='quantile', base='tabm', test=False, genes=None, feature_plan=None):
    """Fit numeric preprocessing on training rows and apply it to the requested rows."""
    if variant != 'raw':
        raise ValueError('This notebook uses the original input features only')
    columns = 168 if base == 'tabm' else 166
    A = data.raw[fit, :columns].copy()
    B = (data.node_x[data.testpos] if test else data.raw[query])[:, :columns].copy()
    X, V, prep = prepare_num(A, B, scaler)
    C = np.empty((len(X), 0), dtype=np.int64)
    D = np.empty((len(V), 0), dtype=np.int64)
    state = dict(variant=variant, base=base, fit_ids_hash=array_hash(data.frame.txId.to_numpy()[fit]), scaler=scaler, numerical_preprocessor=prep, cardinalities=[])
    return (X, V, C, D, state)

def build_net(name, X, cardinalities, config):
    """Construct a neural model from random initial weights."""
    import tabm, rtdl_num_embeddings as emb
    cfg = dict(config)
    cfg.pop('learning_rate')
    cfg.pop('weight_decay')
    bins = emb.compute_bins(torch.from_numpy(X), n_bins=cfg.pop('n_bins'))
    module = emb.PiecewiseLinearEmbeddings(bins, cfg.pop('d_embedding'), activation=False, version='B')
    return tabm.TabM.make(n_num_features=X.shape[1], cat_cardinalities=cardinalities, d_out=2, num_embeddings=module, **cfg).to(DEVICE)

def head_loss(logits, y):
    """Average cross-entropy across all TabM members and training examples."""
    return F.cross_entropy(logits.flatten(0, 1), y.repeat_interleave(logits.shape[1]))

def forward_net(net, name, x, c, pool, y, idx=None):
    return net(x, c if c.shape[1] else None)

@torch.inference_mode()
def net_predict(net, name, V, D, pool, y, batch=256):
    """Run inference and average the 32 member probabilities for TabM."""
    net.eval()
    parts = []
    for start in range(0, len(V), batch):
        x = torch.as_tensor(V[start:start + batch], device=DEVICE)
        c = torch.as_tensor(D[start:start + batch], device=DEVICE)
        logits = forward_net(net, name, x, c, pool, y)
        p = logits.softmax(-1)[..., 1].mean(1)
        parts.append(p.cpu().numpy())
    return np.concatenate(parts)

def fit_net(name, seed, X, C, y, V, D, profile, config=None, val_y=None, epochs=None):
    """Select an epoch count on inner validation, or refit for a fixed number of epochs."""
    seed_all(seed)
    cfg = dict(TABM if config is None else config)
    net = build_net(name, X, [] if C.shape[1] == 0 else [int(max(C.max(), D.max(initial=0))) + 1], cfg)
    tx = torch.as_tensor(X, device=DEVICE)
    tc = torch.as_tensor(C, device=DEVICE)
    ty = torch.as_tensor(y, device=DEVICE)
    lr = cfg['learning_rate']
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=cfg['weight_decay'])
    curve = []
    best = -np.inf
    selected = 0
    stale = 0
    for epoch in range(1, (epochs or profile['max_epochs']) + 1):
        net.train()
        loss_sum = 0.0
        for idx in torch.randperm(len(tx), device=DEVICE).split(profile['batch_size']):
            opt.zero_grad(set_to_none=True)
            logits = forward_net(net, name, tx[idx], tc[idx], tx, ty, idx)
            loss = head_loss(logits, ty[idx])
            if not torch.isfinite(loss):
                raise ValueError('Nonfinite training loss')
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            loss_sum += float(loss.detach()) * len(idx)
        row = dict(epoch=epoch, loss=loss_sum / len(tx))
        if epochs is None:
            if val_y is None:
                raise ValueError('Selection needs inner labels, never outer labels')
            row.update(metrics(val_y, net_predict(net, name, V, D, tx, ty)))
            if row['auc'] is None:
                raise ValueError('Inner selection window has one class')
            if row['auc'] > best:
                best = row['auc']
                selected = epoch
                stale = 0
            else:
                stale += 1
        else:
            selected = epoch
        curve.append(row)
        if epochs is None and stale >= profile['patience']:
            break
    p = net_predict(net, name, V, D, tx, ty)
    checkpoint = dict(state_dict={k: v.detach().cpu().clone() for k, v in net.state_dict().items()}, config=cfg, name=name, input_dim=X.shape[1], cardinalities=[] if C.shape[1] == 0 else [int(C.max()) + 1], epochs=selected, seed=seed, initialization='random', pretrained=False)
    del net, opt, tx, tc, ty
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return (p, selected, curve, checkpoint)

FINAL_ENGINE_HASH = '3661498eb4e7e545253aeae84b480c6d4c349dc7e125920bf97989a5e96c0840'

REPORT_EPOCHS = {'original': {17: 11, 42: 4, 73: 8}, 'balanced': {17: 24, 42: 4, 73: 13}, 'long_inner': {17: 12, 42: 11, 73: 7}}

def submission_audit(data, p, path):
    """Check row order and probability values before saving a submission."""
    p = np.asarray(p, dtype=np.float64)
    if p.shape != (len(data.test),) or not np.isfinite(p).all() or np.any((p < 0) | (p > 1)):
        raise ValueError('Submission probabilities or row count invalid')
    frame = pd.DataFrame({'index': data.sample['index'].to_numpy(), 'target': p})
    if not frame['index'].is_unique:
        raise ValueError('Duplicate submission index')
    np.testing.assert_array_equal(frame['index'], data.test['index'])
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    check = pd.read_csv(path, float_precision='round_trip')
    assert list(check.columns) == ['index', 'target']
    np.testing.assert_array_equal(check['index'], frame['index'])
    np.testing.assert_array_equal(check.target, p)
    return dict(rows=len(frame), columns=list(frame.columns), sha256=file_hash(path), test_ids_hash=array_hash(data.test.txId.to_numpy()), index_matches_test=True, finite_probability_range=True, this_run_kaggle_auc=None, uploaded_to_kaggle=False)

def optimized_submission(data, out, profile, recipe='long_inner', epoch_source='nested_validation'):
    """Select epochs, refit on all labelled training rows and average seeds."""
    if recipe not in REPORT_EPOCHS:
        raise ValueError(recipe)
    if epoch_source not in ['nested_validation', 'reported_fixed']:
        raise ValueError(epoch_source)
    if epoch_source == 'reported_fixed' and profile['mode'] != 'report':
        raise ValueError('Reported epoch counts belong to report mode, not quick checks')
    config = dict(TABM, dropout=0.15 if recipe == 'balanced' else 0.1)
    horizon = 7 if recipe == 'long_inner' else 3
    spec = dict(data=data.identity, profile=profile, recipe=recipe, config=config, inner_horizon=horizon, epoch_source=epoch_source, engine_hash=FINAL_ENGINE_HASH, reported_epochs=REPORT_EPOCHS[recipe])
    job = Path(out) / 'final_tabm' / f'{recipe}_{fingerprint(spec)[:16]}'
    job.mkdir(parents=True, exist_ok=True)
    fit = np.arange(len(data.frame))
    ia, ib = data.split(35 - horizon, horizon)
    parts = []
    records = []
    # Each seed starts from newly initialized weights.
    for seed in profile['seeds']:
        directory = job / f'seed{seed}'
        directory.mkdir(exist_ok=True)
        if (directory / 'complete.json').exists():
            record = json.loads((directory / 'complete.json').read_text())
            assert record['spec'] == json.loads(json.dumps(spec, default=serial))
            assert record['test_ids_hash'] == array_hash(data.test.txId.to_numpy())
            assert record['prediction_hash'] == file_hash(directory / 'test_prediction.npy')
            p = np.load(directory / 'test_prediction.npy')
        else:
            A, B, C, D, inner = make_inputs(data, ia, ib)
            if epoch_source == 'nested_validation':
                _, epochs, selection, _ = fit_net('tabm', seed, A, C, data.y[ia].copy(), B, D, profile, config, val_y=data.y[ib])
            else:
                epochs = REPORT_EPOCHS[recipe][seed]
                selection = [dict(epoch=epochs, source='Predeclared PDF Table A2, not selected using test')]
            save_json(directory / 'selection.json', selection)
            X, V, C, D, outer = make_inputs(data, fit, np.empty(0, dtype=int), test=True)
            p, _, curve, checkpoint = fit_net('tabm', seed, X, C, data.y.copy(), V, D, profile, config, epochs=epochs)
            torch.save(checkpoint, directory / 'model.pt')
            joblib.dump(dict(inner=inner, outer=outer), directory / 'preprocessing.joblib')
            np.save(directory / 'test_prediction.npy', p)
            save_json(directory / 'refit_curve.json', curve)
            record = dict(spec=spec, seed=seed, epochs=epochs, train_rows=len(fit), test_ids_hash=array_hash(data.test.txId.to_numpy()), prediction_hash=file_hash(directory / 'test_prediction.npy'), path=str(directory.resolve()))
            save_json(directory / 'complete.json', record)
        parts.append(p)
        records.append(record)
    probability = np.mean(parts, axis=0)
    filename = 'QUICK_CHECK_ONLY_tabm.csv' if profile['mode'] == 'quick' else f'submission_tabm_optimized_{recipe}.csv'
    path = job / filename
    audit = submission_audit(data, probability, path)
    audit.update(spec=spec, seed_records=records, reported_public_auc=0.96075, reported_public_source='Final PDF section3.4 and appendixH; exact winning artifact missing', exact_096075_submission_reproduced=False, eligible_full_run=profile['mode'] == 'report')
    save_json(job / 'submission_audit.json', audit)
    return (path, audit)


In [ ]:
P = profiles(PROFILE)


## 3. Load and check the data


In [ ]:
# Labelled rows are the only rows used by supervised models.
data = Dataset(DATA_DIR, expected_competition=EXPECTED_COMPETITION)
print('Labelled training rows:', len(data.frame), '| Unknown labels:', int(data.train.label.isna().sum()), '| Test rows:', len(data.test))


Labelled training rows: 31235 | Unknown labels: 110537 | Test rows: 15329


## 4. Choose the final recipe
`long_inner`: dropout 0.1 and the last seven training periods for epoch selection. `balanced`: dropout 0.15 and the last three periods. `original`: dropout 0.1 and the last three periods. These are separate recipes.

Use `nested_validation` to select epochs in the current environment. `reported_fixed` uses Table A2: 12/11/7 epochs for the long-inner seeds 17/42/73. The report associates 0.96075 with longer selection but states that the exact winning configuration and CSV were not retained. This notebook reconstructs the described method; it cannot guarantee that score.


In [ ]:
FINAL_RECIPE = globals().get('FINAL_RECIPE', 'long_inner')
EPOCH_SOURCE = globals().get('EPOCH_SOURCE', 'nested_validation')
assert FINAL_RECIPE in REPORT_EPOCHS
print('Recipe:', FINAL_RECIPE, '| Epoch source:', EPOCH_SOURCE, '| Seeds:', P['seeds'])


Recipe: long_inner | Epoch source: nested_validation | Seeds: [17, 42, 73]


## 5. Select epochs, refit and average probabilities
The long-inner recipe selects epochs using training periods 1-28 and validation periods 29-35. Each seed is then initialized again and trained on all known labels from periods 1-35. Average 32 member probabilities within each seed, then average the seeds. Completed seeds are saved for restart.


In [ ]:
submission, final_audit = optimized_submission(data, OUT, P, FINAL_RECIPE, EPOCH_SOURCE)
display(pd.DataFrame([dict(seed=row['seed'],selected_epochs=row['epochs'],training_rows=row['train_rows'])
    for row in final_audit['seed_records']]))


/usr/local/lib/python3.13/dist-packages/rtdl_num_embeddings.py:340: UserWarning: The 14-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


## 6. Check and download the submission
The output has columns `index,target`, matching the sample submission order. New Kaggle AUC remains empty until the file is submitted and scored.


In [ ]:
frame = pd.read_csv(submission, float_precision='round_trip')
display(frame.head())
print({key: final_audit[key] for key in ['rows','index_matches_test','finite_probability_range','this_run_kaggle_auc']})
print('SHA256:', file_hash(submission))


In [ ]:
DOWNLOAD_CSV = globals().get('DOWNLOAD_CSV', False)
if DOWNLOAD_CSV:
    from google.colab import files
    files.download(str(submission))
else:
    print('Saved to:', submission)
